# 06 — Final Evaluation, Export & Model Summary (Stage 1)

**Purpose:** Load the best available Stage-1 checkpoint, run validation if requested, and export the model.

This notebook does **not** require the full 200-epoch training run. It accepts `best_joint.pth`, `best.pth`, or `latest.pth` from the selected Stage-1 model folder.


In [1]:
import os, sys
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm opencv-python-headless tensorboard onnx onnxruntime

import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 188.4 MB/s eta 0:00:00


In [2]:
import os
import sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import json
import logging
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader

from lib.config import cfg
from lib.models import get_net
from lib.core import get_loss, validate
from lib.dataset import BddDataset
from lib.utils.drive_dataset import (
    ensure_local_dataset_from_drive,
    find_raw_bdd_root,
    resolve_bdd_images_100k_dir,
    resolve_bdd_labels_100k_dir,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Stage1 default final-eval / export target = YOLOPX.
CONFIG = 'YOLOPX'
CHECKPOINT_NAME = 'best_joint.pth'
RUN_EVAL = True
EXPORT_ONNX = True
EXPORT_TORCHSCRIPT = True

# Keep later cells safe even if you re-run them independently after a failed import.
globals().update({
    'CONFIG': CONFIG,
    'CHECKPOINT_NAME': CHECKPOINT_NAME,
    'RUN_EVAL': RUN_EVAL,
    'EXPORT_ONNX': EXPORT_ONNX,
    'EXPORT_TORCHSCRIPT': EXPORT_TORCHSCRIPT,
})

yaml_map = {
    'YOLOPX':             os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopx_vehicle_lane_baseline.yaml'),
    'YOLOP':              os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolop_vehicle_lane_baseline.yaml'),
    'YOLOPv2-paper-no-da': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_paper_no_da.yaml'),
    'YOLOPv2-best-row':   os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_best_row.yaml'),
    'YOLOPv2-focal-only': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_focal_only_ablation.yaml'),
}
run_name_map = {
    'YOLOPX':             'yolopx',
    'YOLOP':              'yolop',
    'YOLOPv2-paper-no-da': 'yolopv2_paper_no_da',
    'YOLOPv2-best-row':   'yolopv2_best_row',
    'YOLOPv2-focal-only': 'yolopv2_focal_only',
}

assert CONFIG in yaml_map, f'Unknown CONFIG={CONFIG}. Valid: {list(yaml_map)}'
yaml_path = yaml_map[CONFIG]
assert os.path.exists(yaml_path), f'Missing config YAML: {yaml_path}'

cfg.defrost()
cfg.merge_from_file(yaml_path)

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)
BDD_IMAGES = resolve_bdd_images_100k_dir(RAW_BDD_ROOT, ECOCAR_ROOT)
BDD_LABELS = resolve_bdd_labels_100k_dir(RAW_BDD_ROOT)

cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = BDD_IMAGES
cfg.DATASET.LABELROOT = BDD_LABELS
cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')

run_name = run_name_map[CONFIG]
cfg.DRIVE.ROOT = ECOCAR_ROOT
cfg.DRIVE.CHECKPOINT_DIR = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'stage1', 'checkpoints', run_name)
cfg.DRIVE.METRICS_DIR    = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'stage1', 'metrics',     run_name)
os.makedirs(cfg.DRIVE.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.DRIVE.METRICS_DIR, exist_ok=True)
os.makedirs(os.path.join(cfg.DRIVE.METRICS_DIR, 'visualization'), exist_ok=True)
cfg.freeze()

# Fail loudly if expected YOLOPX checkpoints are missing — never fall back.
checkpoint_candidates = [
    os.path.join(cfg.DRIVE.CHECKPOINT_DIR, CHECKPOINT_NAME),
    os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best_joint.pth'),
    os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth'),
    os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'latest.pth'),
]
ckpt_path = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
if ckpt_path is None:
    raise FileNotFoundError(
        f'No {CONFIG} checkpoint found in {cfg.DRIVE.CHECKPOINT_DIR}.\n'
        f'Tried: {checkpoint_candidates}\n'
        f'Run notebook 02 first to produce a YOLOPX checkpoint.'
    )

model = get_net(cfg).to(device)
model.gr = 1.0
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

print(f'  postprocess.py : {os.path.exists(os.path.join(REPO_ROOT, "lib", "core", "postprocess.py"))}')
print('=' * 64)
print('[Stage1 / 06 final eval+export preflight]')
print(f'  Model family    : {CONFIG}')
print(f'  Model.NAME      : {cfg.MODEL.NAME}')
print(f'  Config YAML     : {yaml_path}')
print(f'  Checkpoint dir  : {cfg.DRIVE.CHECKPOINT_DIR}')
print(f'  Checkpoint file : {ckpt_path}')
print(f'  Dataset root    : {DATASET_ROOT}')
print(f'  BDD images      : {BDD_IMAGES}')
print(f'  Metrics dir     : {cfg.DRIVE.METRICS_DIR}')
print(f'  Val image size  : {tuple(cfg.TEST.IMAGE_SIZE)}')
print(f'  epoch={ckpt.get("epoch", "?")}  nc={model.nc}  names={model.names}')
print('=' * 64)


Extracting /content/drive/MyDrive/EcoCAR/datasets/bdd100k_vehicle5.tar.gz into this notebook runtime ...
[BDD images] selected: /content/bdd100k_raw/100k | train=70000, val=10000
  postprocess.py : True
[Stage1 / 06 final eval+export preflight]
  Model family    : YOLOPX
  Model.NAME      : YOLOPX
  Config YAML     : /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/configs/yolopx_vehicle_lane_baseline.yaml
  Checkpoint dir  : /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx
  Checkpoint file : /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx/best_joint.pth
  Dataset root    : /content/bdd100k_vehicle5
  BDD images      : /content/bdd100k_raw/100k
  Metrics dir     : /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/metrics/yolopx
  Val image size  : (640, 384)
  epoch=87  nc=1  names=['vehicle']


In [3]:

# ── Full validation first (stage-1 protocol) ──
RUN_EVAL = bool(globals().get('RUN_EVAL', True))
eval_summary = None
if RUN_EVAL:
    val_wh = tuple(getattr(cfg.TEST, 'IMAGE_SIZE', [640, 384]))
    val_size = (int(val_wh[1]), int(val_wh[0]))

    val_dataset = BddDataset(cfg, is_train=False, inputsize=val_size, transform=T.ToTensor())
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.TEST.BATCH_SIZE_PER_GPU,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        collate_fn=val_dataset.collate_fn,
    )

    logger = logging.getLogger(f'eval_{CONFIG}')
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        logger.addHandler(logging.StreamHandler())

    criterion = get_loss(cfg, device)
    output_dir = os.path.join(cfg.DRIVE.METRICS_DIR, 'final_eval')
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'visualization'), exist_ok=True)

    ll_seg_result, det_result, val_loss, maps, times = validate(
        epoch=ckpt.get('epoch', -1),
        config=cfg,
        val_loader=val_loader,
        val_dataset=val_dataset,
        model=model,
        criterion=criterion,
        output_dir=output_dir,
        tb_log_dir='',
        writer_dict=None,
        logger=logger,
        device=device,
    )

    ll_acc, ll_iou, ll_miou = ll_seg_result
    mp, mr, map50, map_all = det_result
    eval_summary = {
        'config': CONFIG,
        'checkpoint': ckpt_path,
        'epoch': int(ckpt.get('epoch', -1)),
        'detection': {
            'precision': float(mp),
            'recall': float(mr),
            'mAP50': float(map50),
            'mAP50_95': float(map_all),
        },
        'lane': {
            'accuracy': float(ll_acc),
            'IoU': float(ll_iou),
            'mIoU': float(ll_miou),
        },
        'timing': {
            'inference_ms_per_image': float(times[0] * 1000.0),
            'nms_ms_per_image': float(times[1] * 1000.0),
        },
        'val_loss': float(val_loss),
    }

    summary_path = os.path.join(output_dir, f'{CONFIG.lower()}_{os.path.basename(ckpt_path).replace(".pth", "")}_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(eval_summary, f, indent=2)

    print('\n' + '=' * 70)
    print('STAGE-1 FINAL VALIDATION SUMMARY')
    print('=' * 70)
    print(json.dumps(eval_summary, indent=2))
    print(f'Saved summary to: {summary_path}')


[Dataset] split=val | layout=explicit_packaged_like
[Dataset] images=/content/bdd100k_raw/100k/val
[Dataset] labels=/content/bdd100k_raw/100k/val
[Dataset] lanes =/content/bdd100k_vehicle5/masks/val
building database...


100%|██████████| 10000/10000 [00:00<00:00, 10202.49it/s]


database build finish: 10000 samples
missing lane masks skipped: 0
missing detection labels: 0


100%|██████████| 313/313 [01:37<00:00,  3.22it/s]


                 all       1e+04    1.08e+05      0.0336       0.931       0.828       0.489
Speed: 2.1/1.6/3.6 ms inference/NMS/total per 640x640 image at batch-size 32
Results saved to /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/metrics/yolopx/final_eval/visualization

STAGE-1 FINAL VALIDATION SUMMARY
{
  "config": "YOLOPX",
  "checkpoint": "/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx/best_joint.pth",
  "epoch": 87,
  "detection": {
    "precision": 0.03362266666666667,
    "recall": 0.9305251893467652,
    "mAP50": 0.8278127978298301,
    "mAP50_95": 0.4892726886576916
  },
  "lane": {
    "accuracy": 0.7919318543206367,
    "IoU": 0.1959031492923778,
    "mIoU": 0.5909761348008542
  },
  "timing": {
    "inference_ms_per_image": 2.073867578376354,
    "nms_ms_per_image": 1.5541543021056663
  },
  "val_loss": 0.33996277804374697
}
Saved summary to: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/metrics/yolopx/final_eval/yolopx_best

In [4]:

import os
EXPORT_ONNX = bool(globals().get('EXPORT_ONNX', True))
EXPORT_TORCHSCRIPT = bool(globals().get('EXPORT_TORCHSCRIPT', True))

try:
    import onnxscript
except ImportError:
    print('onnxscript not found. Installing...')
    !pip install -q onnxscript
    import onnxscript

class DemoExportWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        if hasattr(self.m, 'predict'):
            det_out, lane_prob = self.m.predict(x)
        else:
            det_out, lane_raw = self.m(x)
            if lane_raw.shape[1] == 2:
                lane_prob = torch.softmax(lane_raw, dim=1)[:, 1:2]
            else:
                lane_prob = torch.sigmoid(lane_raw[:, :1])
        if isinstance(det_out, (tuple, list)):
            det_export = det_out[0] if isinstance(det_out, tuple) else det_out[0]
        else:
            det_export = det_out
        return det_export, lane_prob

export_model = DemoExportWrapper(model).to(device).eval()
export_dir = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'exports')
os.makedirs(export_dir, exist_ok=True)

dummy_input = torch.randn(1, 3, 640, 640, device=device)

if EXPORT_ONNX:
    onnx_path = os.path.join(export_dir, f'{CONFIG.lower()}_{os.path.basename(ckpt_path).replace(".pth", "")}.onnx')
    torch.onnx.export(
        export_model,
        dummy_input,
        onnx_path,
        opset_version=18,
        input_names=['images'],
        output_names=['det_out', 'lane_prob'],
        dynamic_axes={
            'images': {0: 'batch'},
            'det_out': {0: 'batch'},
            'lane_prob': {0: 'batch'},
        },
    )
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print(f'ONNX exported to {onnx_path}')

if EXPORT_TORCHSCRIPT:
    ts_path = os.path.join(export_dir, f'{CONFIG.lower()}_{os.path.basename(ckpt_path).replace(".pth", "")}.pt')
    traced = torch.jit.trace(export_model, dummy_input, strict=False, check_trace=False)
    traced.save(ts_path)
    print(f'TorchScript exported to {ts_path}')


onnxscript not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 21.9 MB/s eta 0:00:00


/tmp/ipykernel_2987/1017102713.py:40: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0505 03:04:47.135000 2987 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0505 03:04:47.135000 2987 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0505 03:04:47.136000 2987 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 

[torch.onnx] Obtain model graph for `DemoExportWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DemoExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX exported to /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx/exports/yolopx_best_joint.onnx


/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/lib/models/yolopx_baseline.py:93: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  lane_prob = lane_out[:, 1:2] if lane_out.shape[1] == 2 else lane_out[:, :1]


TorchScript exported to /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx/exports/yolopx_best_joint.pt


In [5]:

# ── Model size summary ──
param_count = sum(p.numel() for p in model.parameters())
ckpt_size_mb = os.path.getsize(ckpt_path) / 1024 / 1024
print(f'Parameters:     {param_count / 1e6:.2f} M')
print(f'Checkpoint:     {ckpt_size_mb:.1f} MB')
print(f'Checkpoint dir: {cfg.DRIVE.CHECKPOINT_DIR}')
print(f'Metrics dir:    {cfg.DRIVE.METRICS_DIR}')
if bool(globals().get('RUN_EVAL', True)) and eval_summary is not None:
    print(f"Eval mAP@0.5:   {eval_summary['detection']['mAP50']:.4f}")
    print(f"Eval lane IoU:  {eval_summary['lane']['IoU']:.4f}")


Parameters:     31.38 M
Checkpoint:     360.0 MB
Checkpoint dir: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx
Metrics dir:    /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/metrics/yolopx
Eval mAP@0.5:   0.8278
Eval lane IoU:  0.1959
